Этап 1. Закидываем промпты в ллм и получаем json файлы: по два на каждое произведение (персонажи, взаимодействия персонажей)

Этап 2. Создаём графы и веб-страницы

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd
import py4cytoscape as p4c #должен быть открыт cytoscape, иначе тут будет ошибка

In [ ]:
source_root = Path(r"C:\Users\nn\Desktop\py\данные для графов") #папка с полученными при помощи ллм json файлами
output_root = Path(r"C:\Users\nn\Desktop\py\данные для страниц")
output_root.mkdir(parents=True, exist_ok=True)

#функция, создающая веб-страницы для графов
def create_webpage_from_cyjs(cyjs_path: Path, output_html_path: Path):
    with open(cyjs_path, 'r', encoding='utf-8') as f:
        cyjs_data = json.load(f)

    elements = cyjs_data.get('elements', {})
    nodes = elements.get('nodes', [])
    edges = elements.get('edges', [])

    degree = {}
    for node in nodes:
        node_id = node['data']['id']
        degree[node_id] = 0
    for edge in edges:
        source = edge['data']['source']
        target = edge['data']['target']
        degree[source] += 1
        degree[target] += 1

    deg_values = list(degree.values())
    min_deg = min(deg_values) if deg_values else 0
    max_deg = max(deg_values) if deg_values else 0

    def map_size(d):
        if max_deg == min_deg:
            return 4
        return 2 + (d - min_deg) * (6 / (max_deg - min_deg))

    nodes_for_js = []
    for node in nodes:
        node_id = node['data']['id']
        node_data = node['data'].copy()
        node_data['degree'] = degree[node_id]
        node_data['size'] = map_size(degree[node_id])
        nodes_for_js.append(node_data)

    edges_for_js = [edge['data'] for edge in edges]

    html_template = """<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Graph</title>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/cytoscape/3.26.0/cytoscape.min.js"></script>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 0;
            padding: 0;
            overflow: hidden;
            height: 100vh;
            display: flex;
            flex-direction: column;
        }}
        #cy {{
            flex: 1;
            background-color: #fafafa;
        }}
        #info-panel {{
            position: absolute;
            bottom: 20px;
            right: 20px;
            width: 320px;
            max-height: 80%;
            overflow-y: auto;
            background: rgba(255, 255, 255, 0.95);
            border-radius: 12px;
            box-shadow: 0 4px 20px rgba(0,0,0,0.15);
            padding: 16px;
            font-size: 13px;
            border: 1px solid #ddd;
            backdrop-filter: blur(4px);
            transition: opacity 0.2s;
            user-select: text;
        }}
        #info-panel table {{
            width: 100%;
            border-collapse: collapse;
        }}
        #info-panel th, #info-panel td {{
            text-align: left;
            padding: 6px 4px;
            border-bottom: 1px solid #f0f0f0;
            vertical-align: top;
        }}
        #info-panel th {{
            font-weight: 600;
            color: #4a4a4a;
            width: 35%;
        }}
        #info-panel td {{
            color: #2c3e50;
            width: 65%;
        }}
        .close-btn {{
            position: absolute;
            top: 8px;
            right: 12px;
            background: none;
            border: none;
            font-size: 18px;
            cursor: pointer;
            pointer-events: auto;
            color: #888;
        }}
        .close-btn:hover {{
            color: #000;
        }}
        h4 {{
            margin: 0 0 8px 0;
            color: #5e4b8b;
        }}
    </style>
</head>
<body>
    <div id="cy"></div>
    <div id="info-panel" style="display: none;">
        <button class="close-btn" id="close-panel">&times;</button>
        <div id="info-content"></div>
    </div>

    <script>
        const nodesData = {nodes_json};
        const edgesData = {edges_json};
        
        const elements = [
            ...nodesData.map(node => ({{ data: node }})),
            ...edgesData.map(edge => ({{ data: edge }}))
        ];
        
        const categoryColors = {{
            'Character': '#dcd0ff',
            'Group': '#cccccc'
        }};
        const defaultColor = '#aaaaaa';
        
        const cy = cytoscape({{
            container: document.getElementById('cy'),
            elements: elements,
            style: [
                {{
                    selector: 'node',
                    style: {{
                        'background-color': function(ele) {{
                            const cat = ele.data('Category');
                            return categoryColors[cat] || defaultColor;
                        }},
                        'width': function(ele) {{ return ele.data('size') || 3; }},
                        'height': function(ele) {{ return ele.data('size') || 3; }},
                        'label': function(ele) {{ return ele.data('display_name') || ele.data('Name') || ele.data('shared name') || ele.data('id'); }},
                        'font-size': '2.2px',
                        'text-valign': 'bottom',
                        'text-halign': 'center',
                        'text-margin-y': 1,
                        'border-width': 0.2,
                        'border-color': '#555'
                    }}
                }},
                {{
                    selector: 'edge',
                    style: {{
                        'width': 0.2,
                        'line-color': '#aaa',
                        'target-arrow-color': '#aaa',
                        'target-arrow-shape': 'triangle',
                        'target-arrow-scale': 0.1,
                        'arrow-scale': 0.1,
                        'curve-style': 'bezier',
                        'label': 'data(relation)',
                        'font-size': '1.5px',
                        'text-rotation': 'autorotate',
                        'text-margin-y': -1,
                        'text-background-opacity': 0.7,
                        'text-background-color': '#ffffff'
                    }}
                }},
                {{
                    selector: 'node.hover-highlight',
                    style: {{
                        'border-width': 0.3,
                        'border-color': '#ffaa00',
                        'overlay-opacity': 0.3,
                        'overlay-color': '#ffaa00',
                        'overlay-padding': '1px'
                    }}
                }},
                {{
                    selector: 'edge.hover-highlight',
                    style: {{
                        'width': 0.4,
                        'line-color': '#ffaa00',
                        'target-arrow-color': '#ffaa00'
                    }}
                }},
                {{
                    selector: '.faded',
                    style: {{
                        'opacity': 0.1
                    }}
                }}
            ],
            layout: {{                                                                                                                                  #НАСТРОЙКИ LAYOUT
                name: 'cose',
                idealEdgeLength: 5,
                padding: 50,
                spacingFactor: 5,
                nodeRepulsion: 40000,
                gravity: 0.0005,
                numIter: 10000
            }},
            wheelSensitivity: 0.5
        }});
        
        let hoveredNode = null;
        function fadeOthers(node) {{
            const neighbors = node.neighborhood().nodes();
            const incidentEdges = node.connectedEdges();
            const highlightSet = node.union(neighbors).union(incidentEdges);
            cy.elements().not(highlightSet).addClass('faded');
            node.addClass('hover-highlight');
            incidentEdges.addClass('hover-highlight');
            neighbors.addClass('hover-highlight');
        }}
        function restoreAll() {{
            cy.elements().removeClass('faded hover-highlight');
        }}
        cy.on('mouseover', 'node', function(evt) {{
            if (hoveredNode === evt.target) return;
            hoveredNode = evt.target;
            fadeOthers(hoveredNode);
        }});
        cy.on('mouseout', 'node', function() {{
            hoveredNode = null;
            restoreAll();
        }});
        
        const tooltipDiv = document.createElement('div');
        tooltipDiv.style.position = 'absolute';
        tooltipDiv.style.backgroundColor = 'rgba(0,0,0,0.7)';
        tooltipDiv.style.color = '#fff';
        tooltipDiv.style.padding = '4px 8px';
        tooltipDiv.style.borderRadius = '6px';
        tooltipDiv.style.fontSize = '12px';
        tooltipDiv.style.pointerEvents = 'none';
        tooltipDiv.style.zIndex = '1000';
        tooltipDiv.style.display = 'none';
        document.body.appendChild(tooltipDiv);
        
        cy.on('mouseover', 'edge', (evt) => {{
        const edge = evt.target;
        const relation = edge.data('relation') || '';
        const sourceNode = cy.getElementById(edge.data('source'));
        const targetNode = cy.getElementById(edge.data('target'));
        const sourceName = (sourceNode.data('display_name') || sourceNode.data('Name') || edge.data('source'));
        const targetName = (targetNode.data('display_name') || targetNode.data('Name') || edge.data('target'));
        tooltipDiv.innerHTML = `${{sourceName}} → ${{targetName}}<br><strong>${{relation}}</strong>`;
        tooltipDiv.style.display = 'block';
        const mouseEvent = evt.originalEvent;
        tooltipDiv.style.left = (mouseEvent.clientX + 10) + 'px';
        tooltipDiv.style.top = (mouseEvent.clientY + 10) + 'px';
    }});

    cy.on('mousemove', 'edge', (evt) => {{
        const edge = evt.target;
        const relation = edge.data('relation') || '';
        const sourceNode = cy.getElementById(edge.data('source'));
        const targetNode = cy.getElementById(edge.data('target'));
        const sourceName = (sourceNode.data('display_name') || sourceNode.data('Name') || edge.data('source'));
        const targetName = (targetNode.data('display_name') || targetNode.data('Name') || edge.data('target'));
        tooltipDiv.innerHTML = `${{sourceName}} → ${{targetName}}<br><strong>${{relation}}</strong>`;
        const mouseEvent = evt.originalEvent;
        tooltipDiv.style.left = (mouseEvent.clientX + 10) + 'px';
        tooltipDiv.style.top = (mouseEvent.clientY + 10) + 'px';
    }});
        const infoPanel = document.getElementById('info-panel');
        const infoContent = document.getElementById('info-content');
        const closeBtn = document.getElementById('close-panel');
        closeBtn.addEventListener('click', () => {{
            infoPanel.style.display = 'none';
        }});
        
        function escapeHtml(str) {{
            if (!str) return '';
            return String(str).replace(/[&<>]/g, function(m) {{
                if (m === '&') return '&amp;';
                if (m === '<') return '&lt;';
                if (m === '>') return '&gt;';
                return m;
            }});
        }}
        
        function formatNodeInfo(nodeData) {{
            var fieldNames = {{
                'display_name': 'Name',
                'Category': 'Category',
                'Description': 'Description',
                'Roles': 'Roles',
                'Epithets': 'Epithets',
                'Attributes': 'Attributes',
                'Actions': 'Actions',
                'AverageShortestPathLength': 'Average Shortest Path Length',
                'ClusteringCoefficient': 'Clustering Coefficient',
                'ClosenessCentrality': 'Closeness Centrality',
                'Degree': 'Degree',
                'BetweennessCentrality': 'Betweenness Centrality',
                'Radiality': 'Radiality'
            }};
            var fieldsOrder = [
                'display_name', 'Category', 'Description', 'Roles', 'Epithets', 'Attributes', 'Actions',
                'AverageShortestPathLength', 'ClusteringCoefficient', 'ClosenessCentrality',
                'Degree', 'BetweennessCentrality', 'Radiality'
            ];
            var title = nodeData.display_name || nodeData.Name || nodeData['shared name'] || nodeData.id || 'Информация об узле';
            var html = '<h4>' + escapeHtml(title) + '</h4>';
            html += '<table>';
            for (var i = 0; i < fieldsOrder.length; i++) {{
                var field = fieldsOrder[i];
                var value = nodeData[field];
                if (value !== undefined && value !== null && value !== '') {{
                    if (typeof value === 'object') value = JSON.stringify(value);
                    var displayName = fieldNames[field] || field;
                    html += '<tr>';
                    html += '<th>' + escapeHtml(displayName) + '</th>';
                    html += '<td>' + escapeHtml(String(value)) + '<\/td>';
                    html += '</tr>';
                }}
            }}
            html += '</table>';
            return html;
        }}
        
        cy.on('tap', 'node', (evt) => {{
            const node = evt.target;
            const nodeData = node.data();
            infoContent.innerHTML = formatNodeInfo(nodeData);
            infoPanel.style.display = 'block';
        }});
        
        document.addEventListener('click', (e) => {{
            if (!infoPanel.contains(e.target) && e.target !== infoPanel && !cy.getElementById(e.target.id)) {{
                infoPanel.style.display = 'none';
            }}
        }});
        
        window.addEventListener('resize', () => {{
            cy.resize();
        }});
    </script>
</body>
</html>"""

    nodes_json = json.dumps(nodes_for_js, ensure_ascii=False, indent=2)
    edges_json = json.dumps(edges_for_js, ensure_ascii=False, indent=2)
    html_content = html_template.format(nodes_json=nodes_json, edges_json=edges_json)

    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"Веб-страница создана: {output_html_path}")


#основной цикл
entries = list(source_root.iterdir())
for i in entries:
    folder_path = Path(i)
    internal_files = list(folder_path.iterdir())
    personas = pd.read_json(internal_files[0])
    relations = pd.read_json(internal_files[1])
    relations = relations[relations['Object'] != '']

    persona_names = set(personas['Name'])
    relation_names = set(relations['Subject']).union(set(relations['Object']))
    names_list = sorted(persona_names.union(relation_names))
    all_names = pd.DataFrame({'Name': names_list})
    all_names['Category'] = all_names['Name'].apply(
        lambda x: 'Character' if x in persona_names else 'Group'
    )
    all_names['id'] = all_names['Name'].astype(str)
    all_names['display_name'] = all_names['Name'].astype(str)

    #оформляем колонки с описаниями
    for col in ['Description', 'Roles', 'Epithets', 'Attributes', 'Actions']:
        all_names[col] = ''

    for _, row in personas.iterrows():
        name = row['Name']
        roles = ', '.join(row['Roles']) if isinstance(row['Roles'], list) else str(row['Roles'])
        epithets = ', '.join(row['Epithets']) if isinstance(row['Epithets'], list) else str(row['Epithets'])
        attributes = ', '.join(row['Attributes']) if isinstance(row['Attributes'], list) else str(row['Attributes'])
        actions = ', '.join(row['Actions']) if isinstance(row['Actions'], list) else str(row['Actions'])
        mask = all_names['Name'] == name
        all_names.loc[mask, 'Description'] = row['Description']
        all_names.loc[mask, 'Roles'] = roles
        all_names.loc[mask, 'Epithets'] = epithets
        all_names.loc[mask, 'Attributes'] = attributes
        all_names.loc[mask, 'Actions'] = actions

    #создаём поле для тултипа
    persona_dict = {}
    for _, row in personas.iterrows():
        name = row['Name']
        roles = ', '.join(row['Roles']) if isinstance(row['Roles'], list) else str(row['Roles'])
        epithets = ', '.join(row['Epithets']) if isinstance(row['Epithets'], list) else str(row['Epithets'])
        attributes = ', '.join(row['Attributes']) if isinstance(row['Attributes'], list) else str(row['Attributes'])
        actions = ', '.join(row['Actions']) if isinstance(row['Actions'], list) else str(row['Actions'])
        info = (f"Description: {row['Description']};\n"
                f"Roles: {roles};\nEpithets: {epithets};\n"
                f"Attributes: {attributes};\nActions: {actions};")
        persona_dict[name] = info
    all_names['description'] = all_names['id'].map(persona_dict).fillna('')

    #создаём сам граф
    edges = pd.DataFrame({
        'source': relations["Subject"].astype(str),
        'target': relations["Object"].astype(str),
        'relation': relations["Relation"]
    })
    p4c.create_network_from_data_frames(nodes=all_names, edges=edges, title='a network')
    p4c.analyze_network()

    #настраиваем внешний вид в cytoscape
    style_name = "myStyle"
    if style_name in p4c.get_visual_style_names():
        p4c.delete_visual_style(style_name)
    defaults = {'NODE_SHAPE': "circle", 'NODE_SIZE': 30, 'EDGE_TRANSPARENCY': 120,
                'NODE_LABEL_POSITION': "W,E,c,0.00,0.00"}
    nodeLabels = p4c.map_visual_property('node label', 'display_name', 'p')
    p4c.create_visual_style(style_name, defaults, [nodeLabels])
    p4c.set_node_color_mapping(table_column='Category',
                               table_column_values=['Character', 'Group'],
                               colors=['#dcd0ff', '#cccccc'],
                               mapping_type='d', style_name=style_name)
    p4c.set_visual_style(style_name)
    p4c.set_edge_tooltip_mapping(table_column='relation', style_name=style_name)
    p4c.set_node_tooltip_mapping('description', style_name=style_name)

    #экспортируем граф
    filename_base = os.path.basename(internal_files[0])
    base_name = filename_base.replace('.json', '')
    cyjs_file = output_root / f"graph-{base_name}.cyjs"
    p4c.export_network(filename=str(cyjs_file), type='cyjs')

    html_target = output_root / f"web_{base_name}.html"
    create_webpage_from_cyjs(cyjs_file, html_target)

    #print(f"готов: {filename_base}\n") 